In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn import set_config
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

import lightgbm as lgb

df = pd.read_csv('../dataset/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [2]:
set_config(transform_output="pandas")

X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = ['ocean_proximity']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)

model = lgb.LGBMRegressor(
    objective='regression',
    learning_rate=0.01,
    n_estimators=2000,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])


pipeline.fit(
    X_train, 
    y_train,
    model__categorical_feature=['cat__ocean_proximity']
)

y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE      35569.2584  45285.1168
MAE       24496.2893  30174.0144
R²            0.9054      0.8435


In [4]:
from scipy.stats import randint, uniform
from sklearn.model_selection import KFold, RandomizedSearchCV
# Presupunând că ai deja definit 'preprocessor' din pasul anterior

base_model = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# 1. Creăm pipeline-ul de bază
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', base_model)
])

# 2. Adăugăm prefixul 'model__' pentru ca RandomizedSearchCV 
# să știe că acești parametri sunt destinați pasului numit 'model'
param_distributions = {
    'model__n_estimators':      randint(200, 2000),
    'model__learning_rate':     uniform(0.005, 0.1),      
    'model__num_leaves':        randint(15, 128),
    'model__max_depth':         randint(3, 12),
    'model__min_child_samples': randint(5, 50),
    'model__subsample':         uniform(0.6, 0.4),       
    'model__colsample_bytree':  uniform(0.6, 0.4),       
    'model__reg_alpha':         uniform(0.0, 1.0),
    'model__reg_lambda':        uniform(0.0, 1.0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Estimatorul devine pipeline-ul
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,                  
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

# 4. Transmitem parametrii suplimentari (categorical_feature) prin .fit()
search.fit(
    X_train, 
    y_train,
    model__categorical_feature=['cat__ocean_proximity']
)

print("Cel mai bun scor CV (RMSE):", -search.best_score_)
print("Cei mai buni parametri:")
for k, v in search.best_params_.items():
    # Curățăm prefixul 'model__' pentru o afișare mai clară
    clean_k = k.replace('model__', '')
    print(f"  {clean_k}: {v}")

# best_pipeline este gata de producție și include atât preprocesarea cât și modelul optim
best_pipeline = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Cel mai bun scor CV (RMSE): 45681.15947457041
Cei mai buni parametri:
  colsample_bytree: 0.6362425738131283
  learning_rate: 0.06683860093330873
  max_depth: 9
  min_child_samples: 7
  n_estimators: 1708
  num_leaves: 65
  reg_alpha: 0.6803075385877797
  reg_lambda: 0.450499251969543
  subsample: 0.6053059844639466


In [5]:
y_pred_train = best_pipeline.predict(X_train)
y_pred_test = best_pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       8839.1934  45613.5172
MAE        6146.6761  30170.9206
R²            0.9942      0.8412


In [6]:
from scipy.stats import randint, uniform
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
import lightgbm as lgb
# Se presupune că 'preprocessor' și 'pipeline' sunt definite din pasul anterior

# 1. Separăm datele de antrenament și validare
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# 2. Transformăm manual setul de validare pentru a-l pregăti de early stopping.
# Preprocesorul se "antrenează" pe X_tr și transformă X_val.
preprocessor.fit(X_tr)
X_val_transformed = preprocessor.transform(X_val)

# 3. Adăugăm prefixul 'model__' pentru căutarea parametrilor
param_distributions = {
    'model__n_estimators':      randint(500, 3000),        
    'model__learning_rate':     uniform(0.005, 0.045),    
    'model__num_leaves':        randint(15, 40),
    'model__max_depth':         randint(3, 8),
    'model__min_child_samples': randint(20, 100),
    'model__subsample':         uniform(0.6, 0.4),
    'model__colsample_bytree':  uniform(0.6, 0.4),
    'model__reg_alpha':         uniform(0.1, 1.9),        
    'model__reg_lambda':        uniform(0.1, 1.9),
}

# 4. Definim parametrii specifici pentru LightGBM folosind prefixul model__
fit_params = {
    # Pasăm setul deja transformat, altfel modelul se blochează pe variabile text
    'model__eval_set': [(X_val_transformed, y_val)], 
    'model__eval_metric': 'rmse',
    'model__callbacks': [lgb.early_stopping(stopping_rounds=50, verbose=False)],
    'model__categorical_feature': ['cat__ocean_proximity']
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 5. Folosim pipeline-ul ca estimator
search = RandomizedSearchCV(
    estimator=pipeline, 
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

# 6. Pasăm fit_params folosind operatorul de despachetare **
search.fit(X_tr, y_tr, **fit_params) 

print("Cel mai bun scor CV (RMSE):", -search.best_score_)
print("Cei mai buni parametri:")
for k, v in search.best_params_.items():
    clean_k = k.replace('model__', '')
    print(f"  {clean_k}: {v}")

best_pipeline = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


c:\Users\CrejenovschiDavid\GitHub Clone\Practica-AI\01-ML-Regression\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Cel mai bun scor CV (RMSE): 46206.30432217337
Cei mai buni parametri:
  colsample_bytree: 0.8196906658824482
  learning_rate: 0.0371568165215028
  max_depth: 7
  min_child_samples: 20
  n_estimators: 1712
  num_leaves: 21
  reg_alpha: 1.348511524150317
  reg_lambda: 0.927778507487993
  subsample: 0.8920157266247274


In [7]:
y_pred_train = best_pipeline.predict(X_train)
y_pred_test = best_pipeline.predict(X_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE      34726.3205  45345.3876
MAE       23218.2140  30278.7318
R²            0.9098      0.8431
